<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter7/Fine-tuning_a_Masked_Language_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning a masked language model (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 3,544 kB of archives.
After this operation, 10.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.3 [3,544 kB]
Fetched 3,544 kB in 2s (1,902 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...


You will need to setup git, adapt your email and name in the following cell.

In [2]:
!git config --global user.email "alexandrupp55@@gmail.com"
!git config --global user.name "Pop123-ux"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [5]:
from transformers import AutoModelForMaskedLM

model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

We can see the nr. of parameters by calling the num_parameters() method:

In [7]:
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> DistilBERT number of parameters: {round(distilbert_num_parameters)}M'")
print(f"'>>> BERT number of parameters: 110M'")

'>>> DistilBERT number of parameters: 67M'
'>>> BERT number of parameters: 110M'


In [8]:
text = "I love [MASK]."

Like BERT, DistilBERT was pretrained on the English Wikipedia and BookCorpus datasets, so we expect the predictions for [MASK] to reflect these domains.

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

With a tokenizer and a model, we can now pass our text example to the model, extract the logits, and print out the top 5 candidates:

In [15]:
import torch

inputs = tokenizer(text, return_tensors='pt')
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_15_tokens = torch.topk(mask_token_logits, 15, dim=1).indices[0].tolist()

for token in top_15_tokens:
  print(text.replace(tokenizer.mask_token, tokenizer.decode([token])))

I love you.
I love him.
I love her.
I love it.
I love myself.
I love them.
I love daddy.
I love lucy.
I love thee.
I love everything.
I love god.
I love this.
I love heaven.
I love that.
I love mommy.


In [14]:
tokenizer.decode(top_10_tokens[0])

'you'

## The dataset ##

To showcase domain adaptation, we’ll use the famous Large Movie Review Dataset (or IMDb for short), which is a corpus of movie reviews that is often used to benchmark sentiment analysis models.

In [18]:
from datasets import load_dataset

imdb_dataset = load_dataset('stanfordnlp/imdb')
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

We can see that the train and test splits each consist of 25,000 reviews, while there is an unlabeled split called unsupervised that contains 50,000 reviews.

Let’s take a look at a few samples to get an idea of what kind of text we’re dealing with.

In [19]:
sample = imdb_dataset['train'].shuffle(seed=42).select(range(3))

for row in sample:
  print(f"\n'>>> Review: {row['text']}")
  print(f"'>>> Label: {row['label']}")


'>>> Review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...
'>>> Label: 1

'>>> Review: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub 

Although we won’t need the labels for language modeling, we can already see that a 0 denotes a negative review, while a 1 corresponds to a positive one.

In [21]:
sample_unsupervised = imdb_dataset['unsupervised'].shuffle(seed=67).select(range(3))
for row in sample_unsupervised:
  print(f">>> Review: {row['text']}")
  print(f">>> Label: {row['label']}")

>>> Review: Based on the stage play by Robert E. Sherwood, The Petrified Forest is a character study revolving around the idea of what it means to be 'American'. Not being American myself, I have to admit that films like this often get on my nerves; but thanks to well defined characters and some great acting; the only thing that this film can be rated as is a true classic. The film takes place mostly in a gas station in the middle of the desert, and what starts out as a rather impromptu tale of romance turns into something more sinister when a gangster on the loose decides to hide out in the station, taking all of its occupant's hostage. The plot centres on the character of Alan Squier; a failed writer looking for a cause. When he happens upon the gas station, he soon notices the tragic 'desert rat', Gabrielle Maple, and it's not long before she's infatuated with his clever observations and mysterious personality. The two share a tangible link, before he decides to leave. But it's not 

As we can see right here, the unsupervised labels are neither 1, nor 0, indicating that this can in fact be used for an unsupervised task, such as clustering

For both auto-regressive and masked language modeling, a common preprocessing step is to concatenate all the examples and then split the whole corpus into chunks of equal size. This is quite different from our usual approach, where we simply tokenize individual examples. Why concatenate everything together? The reason is that individual examples might get truncated if they’re too long, and that would result in losing information that might be useful for the language modeling task!

So to get started, we’ll first tokenize our corpus as usual, but without setting the truncation=True option in our tokenizer. We’ll also grab the word IDs if they are available, as we will need them later on to do whole word masking. We’ll wrap this in a simple function, and while we’re at it we’ll remove the text and label columns since we don’t need them any longer:

In [22]:
def tokenize_function(examples):
  result = tokenizer(examples['text'])
  if tokenizer.is_fast:
    result['word_ids'] = [result.word_ids(i) for i in range(len(result['input_ids']))]
  return result

# Use batched=True to activate fast multithreading!
tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched=True, remove_columns=['text', 'label']
)
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

In [25]:
tokenizer.model_max_length

512

This value is derived from the tokenizer_config.json file associated with a checkpoint;

 in this case we can see that the context size is 512 tokens, just like with BERT.

In [26]:
chunk_size=128

Now comes the fun part. To show how the concatenation works, let’s take a few reviews from our tokenized training set and print out the number of tokens per review:

In [27]:
tokenized_samples = tokenized_datasets['train'][:3]

for idx, sample in enumerate(tokenized_samples['input_ids']):
  print(f"'>>> Review {idx} length: {len(sample)}'")

'>>> Review 0 length: 363'
'>>> Review 1 length: 304'
'>>> Review 2 length: 133'


We can then concatenate all these examples with a simple dictionary comprehension, as follows:

In [28]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples['input_ids'])
print(f"'>>> Concatenated reviews length: {total_length}")

'>>> Concatenated reviews length: 800


In [29]:
chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks['input_ids']:
  print(f"'>>> Chunk length: {len(chunk)}'")

'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 32'


As you can see in this example, the last chunk will generally be smaller than the maximum chunk size. There are two main strategies for dealing with this:

- Drop the last chunk if it’s smaller than chunk_size.
- Pad the last chunk until its length equals chunk_size.

We’ll take the first approach here, so let’s wrap all of the above logic in a single function that we can apply to our tokenized datasets:

In [33]:
def group_texts(examples):
  # Concatenate all texts
  concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
  # Compute length of concatenated texts
  total_length = len(concatenated_examples['input_ids']) # First get the actual length
  total_length = (total_length // chunk_size) * chunk_size # Then adjust for chunking
  # Split by chunks of max_len
  result = {
      k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
      for k, t in concatenated_examples.items()
  }
  # Create a new labels column
  result['labels'] = result['input_ids'].copy()
  return result

Note that in the last step of group_texts() we create a new labels column which is a copy of the input_ids one. As we’ll see shortly, that’s because in masked language modeling the objective is to predict randomly masked tokens in the input batch, and by creating a labels column we provide the ground truth for our language model to learn from.

Let’s now apply group_texts() to our tokenized datasets using our trusty Dataset.map() function:

In [34]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

You can see that grouping and then chunking the texts has produced many more examples than our original 25,000 for the train and test splits. That’s because we now have examples involving contiguous tokens that span across multiple examples from the original corpus. You can see this explicitly by looking for the special [SEP] and [CLS] tokens in one of the chunks:

In [35]:
tokenizer.decode(lm_datasets['train'][1]['input_ids'])

"as the vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men. < br / > < br / > what kills me about i am curious - yellow is that 40 years ago, this was considered pornographic. really, the sex and nudity scenes are few and far between, even then it ' s not shot like some cheaply made porno. while my countrymen mind find it shocking, in reality sex and nudity are a major staple in swedish cinema. even ingmar bergman,"

In [41]:
tokenizer.decode(lm_datasets['train'][1]['labels'])

"as the vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men. < br / > < br / > what kills me about i am curious - yellow is that 40 years ago, this was considered pornographic. really, the sex and nudity scenes are few and far between, even then it ' s not shot like some cheaply made porno. while my countrymen mind find it shocking, in reality sex and nudity are a major staple in swedish cinema. even ingmar bergman,"

As expected from our group_texts() function above, this looks identical to the decoded input_ids — but then how can our model possibly learn anything? We’re missing a key step: inserting [MASK] tokens at random positions in the inputs! Let’s see how we can do this on the fly during fine-tuning using a special data collator.

## Fine-tuning DistilBERT with the Trainer API ##

In [37]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

To see how the random masking works, let’s feed a few examples to the data collator. Since it expects a list of dicts, where each dict represents a single chunk of contiguous text, we first iterate over the dataset before feeding the batch to the collator. We remove the "word_ids" key for this data collator as it does not expect it:

In [38]:
samples = [lm_datasets['train'][i] for i in range(2)]
for sample in samples:
  _ = sample.pop('word_ids')

for chunk in data_collator(samples)['input_ids']:
  print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - [MASK] from my video store because of all the controversy that [MASK] it when it was first released in 1967. [MASK] also heard that at first it was seized [MASK] u [MASK] [MASK]. customs if [MASK] ever tried to enter this country, therefore being a [MASK] of films considered " controversial " i really had to see this for myself. < br / > < br / [MASK] taxis plot is centered around [MASK] young swedish drama student named lena who wants [MASK] learn everything she can about life. [MASK] particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political [MASK] such'

'>>> as the vietnam war and race issues in the united states. in between [MASK] politicians and ordinary denizens of [MASK] about their opinions on politics, she has sex with raw drama [MASK], [MASK], and married men. < br / > < br / > what kills me about i am curious - [MASK] spontaneous that 40 years [MASK], this was 

Nice, it worked! We can see that the [MASK] token has been randomly inserted at various locations in our text. These will be the tokens which our model will have to predict during training — and the beauty of the data collator is that it will randomize the [MASK] insertion with every batch!

One side effect of random masking is that our evaluation metrics will not be deterministic when using the Trainer, since we use the same data collator for the training and test sets. We’ll see later, when we look at fine-tuning with Huggingface Accelerate, how we can use the flexibility of a custom evaluation loop to freeze the randomness.

When training models for masked language modeling, one technique that can be used is to mask whole words together, not just individual tokens. This approach is called whole word masking. If we want to use whole word masking, we will need to build a data collator ourselves. A data collator is just a function that takes a list of samples and converts them into a batch, so let’s do this now! We’ll use the word IDs computed earlier to make a map between word indices and the corresponding tokens, then randomly decide which words to mask and apply that mask on the inputs. Note that the labels are all -100 except for the ones corresponding to mask words.

In [40]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probs = 0.2

def whole_word_masking_data_collator(features):
  for feature in features:
    word_ids = feature.pop('word_ids')

    # Create a map between words and corresponding token indices
    mapping = collections.defaultdict(list)
    current_word_index = -1
    current_word = None
    for idx, word_id in enumerate(word_ids):
      if word_id is not None:
        if word_id != current_word:
          current_word = word_id
          current_word_index += 1
        mapping[current_word_index].append(idx)

    # Randomly mask words
    mask = np.random.binomial(1, wwm_probs, (len(mapping),))
    input_ids = feature['input_ids']
    labels = feature['labels']
    new_labels = [-100] * len(labels)
    for word_id in np.where(mask)[0]:
      word_id = word_id.item()
      for idx in mapping[word_id]:
        new_labels[idx] = labels[idx]
        input_ids[idx] = tokenizer.mask_token_id
    feature['labels'] = new_labels

  return default_data_collator(features)


Next, we can try it on the same samples as before:

In [46]:
samples = [lm_datasets['train'][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch['input_ids']:
  print(f"\n'>>> {tokenizer.decode(chunk)}")


'>>> [CLS] i rented i am curious - [MASK] from my video store [MASK] [MASK] all the controversy that [MASK] it when it [MASK] [MASK] [MASK] in 1967. i also heard [MASK] [MASK] first it was [MASK] by u. s. customs if it ever tried to enter this country, therefore being [MASK] fan of [MASK] considered " controversial " i really had to [MASK] [MASK] for myself. < br / > < br / > the plot [MASK] centered around a young swedish drama [MASK] named lena who wants to learn everything she can about life. in particular she wants to focus her attentions [MASK] making [MASK] sort [MASK] documentary on what the average swede thought about certain political issues such

'>>> as the vietnam war and race issues in the united states. in between asking politicians and [MASK] [MASK] [MASK] [MASK] of stockholm about their opinions [MASK] [MASK], she has sex [MASK] her drama teacher, classmates, and [MASK] men. [MASK] [MASK] / > [MASK] br / > what kills me about i am curious - [MASK] is that [MASK] years 

In [47]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets['train'].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

This has automatically created new train and test splits, with the training set size set to 10,000 examples and the validation set to 10% of that — feel free to increase this if you have a beefy GPU!

Now, we specify the arguments for the Trainer

In [49]:
from transformers import TrainingArguments

batch_size=64
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset['train']) // batch_size
model_name = model_checkpoint.split('/')[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
)

Here we tweaked a few of the default options, including logging_steps to ensure we track the training loss with each epoch. We’ve also used fp16=True to enable mixed-precision training, which gives us another boost in speed. By default, the Trainer will remove any columns that are not part of the model’s forward() method. This means that if you’re using the whole word masking collator, you’ll also need to set remove_unused_columns=False to ensure we don’t lose the word_ids column during training.

In [51]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset['train'],
    eval_dataset=downsampled_dataset['test'],
    data_collator=data_collator,
)

## Perplexity for language models ##

In [52]:
import math

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Step
No log,3.088148,0


>>> Perplexity: 21.94


A lower perplexity score means a better language model, and we can see here that our starting model has a somewhat large value. Let’s see if we can lower it by fine-tuning! To do that, we first run the training loop:

In [53]:
trainer.train()

Step,Training Loss
156,2.681355
312,2.567351
468,2.532907


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.5927608402924425, metrics={'train_runtime': 188.3664, 'train_samples_per_second': 159.264, 'train_steps_per_second': 2.5, 'total_flos': 994208670720000.0, 'train_loss': 2.5927608402924425, 'epoch': 3.0})

and then compute the resulting perplexity on the test set as before:

In [54]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Step
2.532907,2.438533,471


>>> Perplexity: 11.46


Nice — this is quite a reduction in perplexity, which tells us the model has learned something about the domain of movie reviews!

Once training is finished, we can push the model card with the training information to the Hub (the checkpoints are saved during training itself):

## To do: ##
✏️ Your turn! Run the training above after changing the data collator to the whole word masking collator. Do you get better results?

In [55]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/pop123ux/distilbert-base-uncased-finetuned-imdb/commit/a4bfecf692502412e3693c54b3a951c7d66f31b4', commit_message='End of training', commit_description='', oid='a4bfecf692502412e3693c54b3a951c7d66f31b4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pop123ux/distilbert-base-uncased-finetuned-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='pop123ux/distilbert-base-uncased-finetuned-imdb'), pr_revision=None, pr_num=None)

## Fine-tuning DistilBERT with Huggingface Accelerate ##

In [56]:
def insert_random_mask(batch):
  features = [dict(zip(batch, t)) for t in zip(*batch.values())]
  masked_inputs = data_collator(features)
  # Create a new "masked" column for each column in the dataset
  return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

Next, we’ll apply this function to our test set and drop the unmasked columns so we can replace them with the masked ones. You can use whole word masking by replacing the data_collator above with the appropriate one, in which case you should remove the first line here:

In [57]:
downsampled_dataset = downsampled_dataset.remove_columns(['word_ids'])
eval_dataset = downsampled_dataset['test'].map(
    insert_random_mask,
    batched=True,
    remove_columns=downsampled_dataset['test'].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

We can then set up the dataloaders as usual, but we’ll use the default_data_collator from Huggingface Transformers for the evaluation set:

In [58]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

batch_size = 64
train_dataloader = DataLoader(
    downsampled_dataset["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)

In [59]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [64]:
model

DistilBertForMaskedLM(
  (activation): GELUActivation()
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

Then we need to specify the optimizer; we'll use the standard AdamW:

In [65]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=5e-5)

With these objects, we can now prepare everything for training with the Accelerator object:

In [66]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

Now that our model, optimizer, and dataloaders are configured, we can specify the learning rate scheduler as follows:

In [67]:
from transformers import get_scheduler

epochs=3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

In [68]:
from huggingface_hub import get_full_repo_name

model_name = "distilbert-base-uncased-finetuned-imdb-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'pop123ux/distilbert-base-uncased-finetuned-imdb-accelerate'

In [ ]:
from tqdm.auto import tqdm
import torch
import math
from huggingface_hub import upload_folder, create_repo # Added create_repo

output_dir = model_name # Define output_dir
progress_bar = tqdm(range(num_training_steps))

# Ensure the repository exists on Hugging Face Hub
if accelerator.is_main_process:
    create_repo(repo_name, repo_type='model', exist_ok=True)

for epoch in range(epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataset)]
    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
      tokenizer.save_pretrained(output_dir)
      # Changed from accelerator.push_to_hub to upload_folder
      upload_folder(
          repo_id=repo_name,
          folder_path=output_dir, # Use folder_path for upload_folder
          commit_message=f"Training in progress epoch {epoch}"
          # blocking=False is not a parameter for upload_folder, it blocks by default
      )


  0%|          | 0/471 [00:00<?, ?it/s]

## Using our fine-tuned model ##

We can interact with ourfine-tuned model either by using its widget on the Hub or locally with the pipeline from Huggingface Transformers. Let’s use the latter to download our model using the fill-mask pipeline:

In [ ]:
from transformers import pipeline

mask_filler = pipeline(
    "fill-mask", model="pop123ux/distilbert-base-uncased-finetuned-imdb"
)

In [ ]:
preds = mask_filler(text)

for pred in preds:
  print(f">>> {pred['sequence']}")

Neat — our model has clearly adapted its weights to predict words that are more strongly associated with movies!

## To do: ##

✏️ Try it out! To quantify the benefits of domain adaptation, fine-tune a classifier on the IMDb labels for both the pretrained and fine-tuned DistilBERT checkpoints. If you need a refresher on text classification, check out Chapter 3.